In [1]:
from assetextractor.extraction.utils import Config

from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.attributes import ListAttribute, DictAttribute, FileNameAttribute, ListItem

import json
from pathlib import Path
from lxml.etree import tostring

import typing as t
import re

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

In [3]:
class IconCache:
    def __init__(self):
        self.cache : dict[str, FileNameAttribute] = {}
    
    def process(self, asset: Asset) -> dict[str, t.Any]:
        result : dict[str, t.Any] = dict()
        result["guid"] = asset.guid
        result["name"] = asset.find("Standard.Name")()
        icon = asset.find("Standard.IconFilename")
        if isinstance(icon, FileNameAttribute) and icon.is_image:
            self.cache[icon.identifier] = icon
            result["iconPath"] = icon.identifier
    
        if asset.text is not None:
            result["locaText"] = asset.text.values

        return result  

    def process_attribute(self, attr: DictAttribute) -> dict[str, t.Any]:
        result : dict[str, t.Any] = dict()
        result["id"] = attr.name
        result["name"] = attr.name

        icon = attr.find("Icon")
        if isinstance(icon, FileNameAttribute) and icon.is_image:
            self.cache[icon.identifier] = icon
            result["iconPath"] = icon.identifier
    
        if attr.Name() is not None:
            result["locaText"] = attr.Name().values

        return result  

    def to_dict(self):
        result : dict[str, str] = dict()
        for identifier, icon in self.cache.items():
            try:
                data_url = icon.get_data_url(20)
                if data_url is None:
                    raise ValueError()

                result[identifier] = data_url
            except Exception as e:
                print(f"Conversion of {identifier} failed: {e}")

        return result

In [4]:
def flatten_pool(pool: Asset | None) -> list[int]:
    if pool is None:
        return []

    return sorted([asset.guid for asset in pool.pool_assets().keys()])

In [5]:
params: dict[str, t.Any] = dict()
params["constants"] = dict()
icons = IconCache()

schema: dict[str, t.Any] = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Anno 117 Calculator Parameters",
    "description": "Generated schema for Anno 117 calculator parameters",
    "type": "object",
    "properties": {
        "constants": {
            "type": "object",
            "title": "Constants",
            "properties": {},
            "required": []
        }
    },
    "required": []
}

def infer_type_from_value(value: t.Any):
    """Infer JSON schema type from Python value."""
    if isinstance(value, bool):
        return "boolean"
    elif isinstance(value, int):
        return "integer"
    elif isinstance(value, float):
        return "number"
    elif isinstance(value, str):
        return "string"
    elif isinstance(value, list):
        return "array"
    elif isinstance(value, dict):
        return "object"
    else:
        return "string"

def generate_schema_for_value(value: t.Any, path: str=""):
    """Generate schema for a single value."""
    if isinstance(value, dict):
        properties = {}
        required = []
        
        for key, val in value.items():
            properties[key] = generate_schema_for_value(val, f"{path}.{key}")
            if val is not None:
                required.append(key)
        
        local_schema: dict[str, t.Any] = {
            "type": "object",
            "properties": properties
        }
        if required:
            local_schema["required"] = required
        return local_schema
        
    elif isinstance(value, list) and value:
        # Analyze all items in the array to determine schema
        if all(isinstance(item, dict) for item in value):
            # Collect all possible properties from all items
            all_properties = {}
            property_counts = {}
            
            for item in value:
                for prop_key, prop_val in item.items():
                    val_is_valid = prop_val is not None and (not isinstance(prop_val, list) or len(prop_val) > 0)
                    if prop_key not in all_properties and val_is_valid:
                        all_properties[prop_key] = generate_schema_for_value(prop_val, f"{path}[].{prop_key}")
                        property_counts[prop_key] = 0
                    if val_is_valid:
                        property_counts[prop_key] += 1
            
            # Determine required properties (present in all items)
            required_properties = [prop for prop, count in property_counts.items() if count == len(value)]
            
            item_schema: dict[str, t.Any] = {
                "type": "object",
                "properties": all_properties
            }
            if required_properties:
                item_schema["required"] = required_properties
        else:
            # For non-dict arrays, use the first item as before
            item_schema = generate_schema_for_value(value[0], f"{path}[0]")
        
        return {
            "type": "array",
            "items": item_schema
        }
        
    else:
        return {"type": infer_type_from_value(value)}

def add_constant(key: str, value: t.Any, description: str):
    params["constants"][key] = value
    schema["properties"]["constants"]["properties"][key] = {"type": infer_type_from_value(value), "description": description}
    schema["properties"]["constants"]["required"].append(key)

def add_parameter(key: str, value: list[t.Any] | dict[str,t.Any], class_name: str, description: str | None = None):
    """Add a parameter to the params dict with metadata."""    
    
    schema["properties"][key] = generate_schema_for_value(value, key)
    schema["properties"][key]["title"] = class_name
    schema["required"].append(key)
                
    if  description:
        schema["properties"][key]["description"] = description
    
    params[key] = value

### Constants

In [6]:
fuel = templates["EconomyFeature"].assets[0].EconomyFeature7.Fuel.Products[0]
add_constant("fuelProductionTime", fuel.ProductionTime().seconds, fuel.ProductionTime.meta.description + " in seconds")
add_constant("fuelProduct", fuel.FuelProduct.guid, fuel.FuelProduct.meta.description)

### Languages

In [7]:
add_parameter("languages", [l.lower() for l in assets.texts.languages.literals], "Language", "Supported localization languages")
#add_parameter("languages", ["english"], "Language", "Supported localization languages")

### Need Consumption

In [8]:
needConsumption = templates["DifficultyBalancing"].assets[0].DifficultySettings.NeedConsumption
factories = [{"id": cfg.name, "name": cfg.name, "locaText": cfg.ValueName().values, "consumptionFactor": cfg.ConsumptionFactor()} for cfg in needConsumption]
add_parameter("needConsumptions", factories, "NeedConsumption", "Scaling factor for NeedConsumptionRate")

### Regions

In [9]:
add_parameter("regions", [{**icons.process(asset), "id": asset.Region.RegionID()} for asset in templates["Region"].assets], "Region", "Game regions data with GUID, name, icon, and localized text")

### Sessions (Require manual adjustment to include new ones)

In [10]:
#add_parameter("sessions", [icons.process(assets[guid]) for guid in [3225, 6627]], "Session", "Game session information including region associations")
result: list[t.Any] = []
regions_without_sessions: set[int] = set()
for region in params["regions"]:
    regions_without_sessions.add(region["guid"])

for guid in [37135, 3245, 6627]:
    asset = assets[guid]

    if asset is None:
         raise ValueError(f"Session with GUID {guid} not found") 

    session = icons.process(asset) 
    session["region"] = asset.find("Session.Region").guid
    regions_without_sessions.remove(session["region"])
    result.append(session)

add_parameter("sessions", result, "Session", "Game session information including region associations")

if len(regions_without_sessions) != 0:
    for guid in regions_without_sessions:
        region = next(r for r in params["regions"] if r["guid"] == guid)
        print(f"Region without session: GUID {guid}, Name: {region['name']}")

### Attributes

In [11]:
dict_attr: DictAttribute = templates["NeedAttributeFeature"].assets[0].NeedAttributeFeature.NeedAttributeConfig
result = [icons.process_attribute(attr) for attr in dict_attr]
add_parameter("needAttributes", result, "NeedAttribute", "Attributes obtained from needs")

### Needs

In [12]:
dict_attr: DictAttribute = templates["NeedCategoryConfig"].assets[0].NeedCategoryConfig.NeedCategorys
result = [icons.process_attribute(attr) for attr in dict_attr]
add_parameter("needCategories", result, "NeedCategory", "Categories for needs")

In [13]:
result = []
for asset in templates["Need"].assets:
    js = icons.process(asset)
    cfg = asset.Need
    js["needProduct"] = cfg.NeedProduct.guid
    js["needCategory"] = cfg.NeedCategoryType()
    js["supplyWeight"] = cfg.SupplyWeight()
    js["isBuilding"] = cfg.NeedProduct().Product.IsAbstract()
    js["needAttributes"] = dict()
    for attr in cfg.NeedAttributes:
        js["needAttributes"][attr.name] = int(attr.Value())
    result.append(js)
add_parameter("needs", result, "Need", "Population group definitions with associated levels")

### Population Groups, Levels, and Residences

In [14]:

result = []
for asset in templates["PopulationGroup7"].assets:
    js = icons.process(asset)
    
    js["populationLevels"] = [level.Level.guid for level in asset.find("PopulationGroup7.PopulationLevels")]
    js["region"] = asset.find("PopulationGroup7.Regional")()

    result.append(js)
add_parameter("populationGroups", result, "PopulationGroup", "Population group definitions with associated levels")

In [15]:
result = []
level_to_region : dict[int, str]= dict()
for asset in templates["ResidenceBuilding"].assets:
    js = icons.process(asset)
    

    js["associatedRegions"] = asset.Building.AssociatedRegions()
    js["possibleUpgrades"] = [upgrade.UpgradeGUID.guid for upgrade in asset.Upgradable.PossibleUpgrades]
    
    cfg = asset.Residence7
    js["populationLevel"] = cfg.PopulationLevel.guid

    level_to_region[js["populationLevel"]] = js["associatedRegions"]

    js["needsList"] = []
    for need in cfg.NeedsList:
        js["needsList"].append({
            "need": need.Need.guid,
            "needConsumptionRate": None if need.NeedConsumptionRate() == 0 else need.NeedConsumptionRate()
        })

    result.append(js)
add_parameter("residenceBuildings", result, "ResidenceBuilding", "Residence building for a population level")

### Population Levels

In [16]:
result = []
for asset in templates["PopulationLevel"].assets:
    js = icons.process(asset)
    
    cfg = asset.PopulationLevel
    js["connectedWorkforce"] = cfg.ConnectedWorkforce.guid
    js["populationToWorkforceFactor"] = cfg.PopulationToWorkforceFactor()
    js["associatedRegions"] = level_to_region[js["guid"]]
    result.append(js)
add_parameter("populationLevels", result, "PopulationLevel", "Population level definitions with needs and requirements")

### Products and Workforce

In [17]:

products = []
workforce = []
for asset in templates["Product"].assets:
    if asset.Product.IsWorkforce():
        workforce.append({**icons.process(asset), "associatedRegions": asset.Product.AssociatedRegion()})
    else:
        category = asset.Product.ProductCategory()
        products.append({**icons.process(asset), 
        "associatedRegions": asset.Product.AssociatedRegion(), 
        "isAbstract": asset.Product.IsAbstract(),
        "isConstructionMaterial": category is not None and category() == "Construction Material" and asset.guid != 2180})

add_parameter("products", products, "Product", "Product definitions with producer information")
add_parameter("workforce", workforce, "Workforce", "Product definitions with producer information")

### Product Filter Meta

In [18]:
result = []

# First, collect all products from category index 0 (the "all products" category)
all_products_category = assets[28749].ProductFilter.Categories[0]
all_products = set(p.Product.guid for p in all_products_category.ProductList().ProductList.List)

# Collect products already assigned to categories
assigned_products = set()
for c in assets[28749].ProductFilter.Categories:
    if c.index == 0:
        continue

    category = {}
    category["iconPath"] = icons.process(c.Icon())["iconPath"]
    category["locaText"] = c.Text().values
    category["guid"] = c.ProductList.guid
    category["products"] = [p.Product.guid for p in c.ProductList().ProductList.List]
    
    # Track which products are already assigned
    assigned_products.update(category["products"])
    
    result.append(category)

# Find missing products (in all_products but not in any category)
missing_products = all_products - assigned_products

# Check other ProductList templates for additional products
for asset in templates["ProductList"].assets:
    if asset.guid != 28749:  # Skip the main ProductFilter asset
        other_products = set(p.Product.guid for p in asset.ProductList.List)
        # Add products that aren't already assigned to any category
        missing_products.update(other_products - assigned_products)

# Print missing products with their details
print("Missing products:")
for guid in sorted(missing_products):
    product_asset = assets[guid]
    if product_asset is not None:
        name = product_asset.find("Standard.Name")()
        english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
        print(f"  GUID {guid}: {name} - {english_text}")

add_parameter("productFilters", result, "ProductFilter", "Product category filters for UI organization")

def add_missing(guid: int, category_index: int):
    if not guid in missing_products:
        return
    
    product_asset = assets[guid]
    if product_asset is not None:
        name = product_asset.find("Standard.Name")()
        english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
        result[category_index]['products'].append(guid)
        print(f"Appended {guid}: {name} - {english_text} to {result[category_index]['locaText']['english']}")


Missing products:


In [19]:
#add_missing(2166, 0)
#add_missing(31701, 3)

### Factories

In [20]:
factories = []
modules = []
producers: dict[int, list[int]] = {}

# Track products used in factories
factory_input_products = set()
factory_output_products = set()

def process_list(lis: ListAttribute):
    res: list[dict[str, int]] = []

    for item in lis:
        res.append({
            "product": item.Product.guid,
            "amount": item.Amount()
        })
    
    return res

for template in templates.groups["Objects"]["Buildings"]["Factories"]:
    if not isinstance(template, Template) or template.name == "Monument":
        continue
    
    for asset in template.assets:

        if "TEST" in asset.name:
            continue

        factory = icons.process(asset)
        factory["associatedRegions"] = asset.find("Building.AssociatedRegions")()

        inputs = asset.find("FactoryBase.FactoryInputs")
        factory["inputs"] = process_list(inputs)

        # Track input products
        for input_item in factory["inputs"]:
            factory_input_products.add(input_item["product"])

        factory["needsFuelInput"] = asset.find("FactoryBase.NeedsFuelInput")()

        outputs = asset.find("FactoryBase.FactoryOutputs")
        factory["outputs"] = process_list(outputs)

        # Track output products
        for output_item in factory["outputs"]:
            factory_output_products.add(output_item["product"])

        for o in factory["outputs"]:
            guid = o["product"]
            if guid in producers:
                producers[guid].append(asset.guid)
            else:
                producers[guid] = [asset.guid]

        factory["maintenances"] = process_list(asset.Maintenance.Maintenances)

        factory["cycleTime"] = asset.find("FactoryBase.CycleTime")()

        modules_limit = asset.find("ModuleOwner.ModuleLimits.Main.Limit")
        factory["modulesLimit"] = 0 if modules_limit is None else modules_limit()

        attr_aqueduct = asset.find("AqueductConsumer.AqueductConsumptionBuffEffect")
        if attr_aqueduct is not None and len(attr_aqueduct._value_list) > 0:
            factory["aqueductProductivityBuff"] = attr_aqueduct[0].ProductivityBuff.guid

        effect_attr = asset.find("Building.FunctionalEffects")
        if effect_attr is not None and len(effect_attr._value_list) > 0:
            factory["buffs"] = [buff.GUID.guid for item in effect_attr for buff in item.FunctionalEffect().Effect.Buffs ]

        module_attr = asset.find("ModuleOwner.AdditionalModule")
        if module_attr is not None:
            factory["additionalModule"] = module_attr.guid

        if template.name == "ProductionModuleSilo":
            modules.append(factory);
        else:
            if len(factory["outputs"]) == 0:
                raise ValueError(f"Factory {str(asset)} with GUID {asset.guid} of template {template.name} produces no goods.")
            
            factories.append(factory)

add_parameter("factories", factories, "Factory", "Factory building definitions with inputs, outputs, maintenances, and production rates")
add_parameter("modules", modules, "Module", "Modules attachable to factories with inputs, maintenances, and production rates")

for product in params["products"]:
    if product["guid"] in producers:
        factories = producers[product["guid"]]
        product["mainFactory"] = factories[0]
        product["producers"] = factories

# Check which factory products are not in product filters
all_factory_products = factory_input_products | factory_output_products
filter_products = set()
for category in params["productFilters"]:
    filter_products.update(category["products"])

missing_from_filters = all_factory_products - filter_products

if len(missing_from_filters):
    print("Factory products not in product filters:")
    for guid in sorted(missing_from_filters):
        product_asset = assets[guid]
        if product_asset is not None:
            name = product_asset.find("Standard.Name")()
            english_text = product_asset.text.values.get("english", "") if product_asset.text else ""
            usage = []
            if guid in factory_input_products:
                usage.append("input")
            if guid in factory_output_products:
                usage.append("output")
            print(f"  GUID {guid}: {name} - {english_text} (used as: {', '.join(usage)})")

### Building Buffs and Effects

In [21]:
# print potentially new buffs or properties
diff = set(prop.name for prop in templates.groups["EffectSystem"]["Buffs"] if len(prop.assets)) - set(['TroopBuff', 'MetaBuff', 'DefenseBuildingBuff', 'AreaBuff', 'ShipBuff', 'BuildingBuff'])
if len(diff):
    print("New buff type", diff)

diff = set(prop.name for prop in templates["BuildingBuff"]) - set(['BuildingUpgrade', 'CityInstitutionUpgrade', 'IncidentInfectableUpgrade', 'Text', 'ModuleOwnerUpgrade', 'Standard', 'MaintenanceUpgrade', 'Buff', 'DistributionUpgrade', 'HealthUpgrade', 'IrrigationUpgrade', 'AqueductUpgrade', 'ResidenceUpgrade', 'WarehouseUpgrade', 'RecruitmentUpgrade', 'FactoryUpgrade'])
if len(diff):
    print("New BuildingBuff property", diff)

diff = set(prop.name for prop in assets.properties["BuildingUpgrade"]) - set(['AdditionalFunctionalEffect', 'AdditionalAttributes', 'AttributeModifierInPercent', 'AdditionalWorkforces', 'WorkforceModifierInPercent'])
if len(diff):
    print("New BuildingUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["ResidenceUpgrade"]) - set(['ConsumptionModifierInPercent',
 'GoodConsumptionUpgrade',
 'NeedProvidedNeedAttributes',
 'ProvidedNeedUpgrade'])
if len(diff):
    print("New ResidenceUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["FactoryUpgrade"]) - set(['AddedFertility', 'InputAmountUpgrade', 'CanUseMarsh', 'NeededAreaUpgrade', 'AdditionalOutput', 'FertilityPercent', 'InfluenceRadiusUpgrade', 'CanUseForest', 'ReplaceInputs', 'ProductivityUpgrade', 'FuelDurationPercent', 'CanUseMeadow'])
if len(diff):
    print("New FactoryUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["ModuleOwnerUpgrade"]) - set(['ModuleLimitPercent'])
if len(diff):
    print("New ModuleOwnerUpgrade attribute", diff)

diff = set(prop.name for prop in assets.properties["MaintenanceUpgrade"]) - set(['ReplaceWorkforce', 'MaintenanceFactorUpgrade', 'WorkforceMaintenanceFactorUpgrade', 'EncampedUnitScalingFactorUpgrade'])
if len(diff):
    print("New MaintenanceUpgrade attribute", diff)

In [22]:
for prop in ["BuildingUpgrade","ResidenceUpgrade","FactoryUpgrade", "MaintenanceUpgrade"]:
    assets.properties[prop].print_tree()

BuildingUpgrade:
	AdditionalFunctionalEffect: Asset (Adds another functional effect on target buildings, should use same scope and targets as base functional effect!)
	AdditionalAttributes: 
		AmountOrPercent: FloatOrPercental
	AttributeModifierInPercent: Float (modifies all need attribute outputs)
	WorkforceModifierInPercent: Float (How much the workforce output should increase in percent. (0 = no increase, 100 = double the workforce, etc.))
	AdditionalWorkforces: (population generates the same amount of workforce for other workforce types)
		WorkforceGUID: Asset
*) inherited **) default
ResidenceUpgrade:
	ProvidedNeedUpgrade: (Saturation of public service need ProvidedNeed is always at 100%)
		ProvidedNeed: Asset (only public service needs are allowed)
	GoodConsumptionUpgrade: (Reduces the need for a specific good by X%)
		ProvidedNeed: Asset
		AmountInPercent: Float (The corresponding PopulationInput value in the Population Level is affected by this value. Meaning the max required i

In [23]:
from assetextractor.parsing.core.attributes import PrimitiveAttribute,UpgradeAttribute
ignored_buffs = [95767, # Cheat Boost
                51013] # Maria Bacca (no loca or icon)

# hardcoded buffs
baseProductivityUpgrade = {
    43610: 50
}

result = []
relevant_buffs : set[int] = set()

# upgrade properties
upgrade_paths = {
    "BuildingUpgrade.AdditionalWorkforces": "additionalWorkforces",
    "BuildingUpgrade.WorkforceModifierInPercent": "workforceModifierInPercent",
    "FactoryUpgrade.AdditionalOutput": "additionalOutputs",
    "FactoryUpgrade.ProductivityUpgrade": "productivityUpgrade",
    "FactoryUpgrade.FuelDurationPercent": "fuelDurationPercent",
    "FactoryUpgrade.ReplaceInputs": "replaceInputs",
    "MaintenanceUpgrade.ReplaceWorkforce": "replaceWorkforce",
    "MaintenanceUpgrade.WorkforceMaintenanceFactorUpgrade": "workforceMaintenanceFactorUpgrade"
}

unhandeled_paths = [
    "ResidenceUpgrade.GoodConsumptionUpgrade",
    "FactoryUpgrade.InputAmountUpgrade",
    "FactoryUpgrade.MaxWorkerAmountUpgrade",
]

for asset in templates["BuildingBuff"].assets:
    if asset.guid in ignored_buffs:
        continue

    icon = asset.find("Standard.IconFilename")
    if not isinstance(icon, FileNameAttribute) or not icon.is_image:
        continue
    
    # Check for relevant properties
    has_relevant_properties = False  

    if asset.guid in baseProductivityUpgrade:
        has_relevant_properties = True
    
    # Check each path for meaningful values
    for path, key in upgrade_paths.items():
        try:
            attribute = asset.find(path)
            if attribute is not None:
                if isinstance(attribute, ListAttribute) and len(attribute._value_list) > 0:
                    has_relevant_properties = True
                    break
                elif (isinstance(attribute, PrimitiveAttribute) or isinstance(attribute, UpgradeAttribute)) and attribute() != 0.0:
                    if not (key == "productivityUpgrade" and attribute() < 0 or # Exclude negative productivity upgrades from catastrophe events
                            key == "workforceModifierInPercent" and attribute() > 0): # Exclude more workforce required from catastrophe events
                        has_relevant_properties = True
                        break
                elif "OldWorkforce" in attribute and attribute.OldWorkforce(): # ReplaceWorkforce
                    has_relevant_properties = True
                    break
        except Exception as e:
            print(asset, asset.guid, e)
            continue

    if not str(asset).startswith("Nearby"): # Exclude effects that exist to fullfill needs from public services
        for path in unhandeled_paths:
            attribute = asset.find(path)
            if attribute is not None:
                if isinstance(attribute, ListAttribute) and len(attribute._value_list) > 0:
                    print(asset, asset.guid, path, attribute(), "not handeled")
                elif (isinstance(attribute, PrimitiveAttribute) or isinstance(attribute, UpgradeAttribute)) and attribute() != 0.0:
                    has_relevant_properties = True
                    print(asset, asset.guid, path, attribute(), "not handeled")

    
    
    if has_relevant_properties:
        buff = icons.process(asset) 
        buff["isStackable"] = asset.Buff.IsStackable() or buff["guid"] == 82018 # deep mines

        buff["baseProductivityUpgrade"] = baseProductivityUpgrade[asset.guid] if asset.guid in baseProductivityUpgrade else 0

        # Add building upgrade properties
        for path, key in upgrade_paths.items():
            try:
                attribute = asset.find(path)

                if key == "additionalWorkforces":
                    buff[key] = [workforce.WorkforceGUID.guid for workforce in attribute]
                elif key == "additionalOutputs":
                    buff[key] = [{
                        		"product": extra.Product.guid,
                                "forceProductSameAsFactoryOutput": extra.ForceProductSameAsFactoryOutput(),
                                "additionalOutputCycle": extra.AdditionalOutputCycle(), 
                                "amount": extra.Amount() 
                    } for extra in attribute]
                elif key == "replaceInputs":
                    buff[key] = [{
                        		"newInput": entry.NewInput.guid,
                                "oldInput": entry.OldInput.guid,
                    } for entry in attribute]
                elif key == "replaceWorkforce":
                    buff[key] = {
                        		"newWorkforce": attribute.NewWorkforce.guid,
                                "oldWorkforce": attribute.OldWorkforce.guid,
                    }
                else:
                    buff[key] = attribute()
            except:
                continue
  
        relevant_buffs.add(asset.guid)
        result.append(buff)

add_parameter("buildingBuffs", result, "BuildingBuff", "Building buff assets with upgrade properties for buildings, residences, and factories. Determine what the effect is. See effects for where they apply.")

schema["properties"]["buildingBuffs"]["items"]["properties"]["replaceInputs"] = generate_schema_for_value([{
    "newInput": 1,
    "oldInput": 1
}])

Amphitheatre Event (temporary) 51356 ResidenceUpgrade.GoodConsumptionUpgrade [Item (0)] not handeled
False Flames and Epidemics 89265 ResidenceUpgrade.GoodConsumptionUpgrade [Item (0)] not handeled
Bread Sentries 140884 ResidenceUpgrade.GoodConsumptionUpgrade [Item (0)] not handeled


In [24]:
ignored_effects = [
    82990, # workfoce upgrade for warehouses
    107478, 107622, # Workforce increase (bad outcome from event)
    82164, 97812, # clever carpentry and hand and heart (removed techs)
    118552, # unused Mercury Festival Gold Production
    97795, # Effect Productivity Hand and Heart deperecated 
    49804 # Effect Prydein Legendary Mines (removed)
]
relevant_effects : set[int] = set()
result= []
effect_json : dict[int,any] = dict()

regex_all_buildings = r".*All (Production )?Buildings.*"

for asset in templates["Effect"].assets:
    if asset.guid in ignored_effects:
        continue

    try:
        cfg = asset.Effect
        buffs = [buff.GUID.guid for buff in cfg.Buffs if buff.GUID.guid  in relevant_buffs]
        if len(buffs):

            js = icons.process(asset)
            js["buffs"] = buffs
            js["targets"] = list(dict.fromkeys(guid for pool in cfg.Targets for guid in flatten_pool(pool.find_ref("GUID"))))
            if (len(js["targets"]) == 0):
                continue 

            js["targetsIsAllProduction"] = any([re.match(regex_all_buildings, pool.GUID().Standard.Name()) for pool in cfg.Targets if pool.GUID()])
            if js["targetsIsAllProduction"]:
                print(js["guid"], js["name"])
            js["effectScope"] = cfg.EffectScope()
            js["excludeEffectSourceGUID"] = cfg.ExcludeEffectSourceGUID()
            js["effectDuration"] = cfg.TimedEffect.EffectDuration().seconds

            if len(cfg.ExcludeFromTargets._value_list) > 0:
                print(asset.name, "ExcludeFromTargets", cfg.ExcludeFromTargets())

            js["source"] = "session-event" if js["effectScope"].endswith("Session") else ("module" if js["effectScope"] == "ModuleOwner" else "island-event")


            relevant_effects.add(asset.guid)
            effect_json[asset.guid] = js
            result.append(js)
    except Exception as e:
        print(asset.guid, e)

add_parameter("effects", result, "Effect", "Building buff combined with targets (pool of building GUIDs). Source is an enum with literals 'module', 'tech', 'festival', 'veneration-effect', 'session-even', 'island-event'")

### Techs (must be after buffs)

In [25]:
result = []
for asset in templates["Tech"].assets:   
    
    cfg = asset.Tech    

    effects = [effect.EffectAsset.guid for effect in cfg.Rewards.Effects if effect.EffectAsset.guid  in relevant_effects]
    if len(effects):
        js = icons.process(asset)
        js["locaText"] = cfg.TechName().values
        js["effects"] = effects
        js["isRepeatable"] = cfg.IsRepeatable()

        for effect_guid in effects:
            effect_json[effect_guid]["source"] = "tech"

        result.append(js)
add_parameter("techs", result, "Tech", "Technologies from the discovery tree with relevant effects")

### Religion

In [26]:
result = []
for asset in templates["Patron"].assets:   
    
    cfg = asset.Patron 
    js = icons.process(asset)
    js["locaText"] = cfg.PatronName().values   
    js["wonder"] = cfg.Wonder.guid if cfg.Wonder.guid in relevant_effects else None
    if js["wonder"] in effect_json:
        effect_json[js["wonder"]]["effectScope"] = "ObjectsInMeta"
        effect_json[js["wonder"]]["source"] = "veneration-effect"
    js["dominantEffects"] = [item.GUID.guid for item in cfg.DominantEffects if item.GUID.guid in relevant_effects]
    js["localEffects"] = []
    for entry in cfg.LocalEffects:
        if entry.GUID.guid not in relevant_effects:
            continue

        js["localEffects"].append({
            "effect": entry.GUID.guid,
            "milestones": [{
                "devotion": item.Devotion(),
                "buffScaling": item.BuffScaling()
            } for item in entry.Milestones]
        })
        

    result.append(js)
add_parameter("patrons", result, "Patrons", "Patrons with effects increased by devotion.")

### Festivals

In [27]:
for asset in templates["Festival"].assets:  
    try:
        cfg = asset.Festival
        effects = [effect.Effect.guid for effect in cfg.FestivalEffects if effect.Effect.guid  in relevant_effects]
        if len(effects):
            print(asset.Standard.Name)
            
            duration = cfg.FestivalDuration().seconds
            for effect in effects:
                js = effect_json[effect]
                js["effectDuration"] = duration
                js["source"] = "festival"

    except Exception as e:
        print(asset.guid, e)

Name: Festival Cernunnos
Name: Festival Minerva


### Items

In [28]:
result = []
for template in ["Item", "ItemWithBoost"]:
    for asset in templates[template].assets:
        try:
            cfg = asset.Effect
            buffs = [buff.GUID.guid for buff in cfg.Buffs if buff.GUID.guid  in relevant_buffs]
            if len(buffs):

                js = icons.process(asset)
                js["buffs"] = buffs
                js["targets"] = [guid for pool in cfg.Targets for guid in flatten_pool(pool.GUID())]
                
                if len(js["targets"]) == 0:
                    print(asset, asset.guid, "has no targets")

                js["effectScope"] = cfg.EffectScope()
                js["excludeEffectSourceGUID"] = cfg.ExcludeEffectSourceGUID()
                js["rarity"] = asset.Item.Rarity()

                if len(cfg.ExcludeFromTargets._value_list) > 0:
                    print(asset, asset.guid, "ExcludeFromTargets", cfg.ExcludeFromTargets())

                #if not asset.Item.OnlyEquippableOnce():
                #    print(asset, asset.guid, "OnlyEquippableOnce", asset.Item.OnlyEquippableOnce())

                relevant_effects.add(asset.guid)
                result.append(js)
        except Exception as e:
            print(asset.guid, e)

add_parameter("items", result, "Item", "Items equipable in Villa with relevant buffs.")

Elephant Handler 54336 has no targets
Elephant Handler 54337 has no targets
Elephant Handler 44701 has no targets
Elephant Handler 44702 has no targets
Specialist Onboarding Production 68050 has no targets


### Icons (after all other parameters are processed)

In [29]:
add_parameter("icons", icons.to_dict(), "Icon", "Icon data URLs indexed by icon identifier")

In [30]:
# Override the icons schema to use additionalProperties instead of individual properties
schema["properties"]["icons"] = {
    "type": "object",
    "title": "Icon",
    "description": "Icon data URLs indexed by icon identifier",
    "additionalProperties": {
        "type": "string"
    }
}

## Texts

In [31]:
result = []

for referenceName, lineID in {
    "activeIslandEffects": -6901888795121698763,
    "affectedBuildings": -6904082780390519730,
    "all": -6915762395677959303,
    "allAreaEffects": -6911533831834980623,
    "allBuildings": -6905197213145912196,
    "allIslands": -6901911865748271091,
    "apply": -6911313214704811434,
    "areaEffects": -6903712981431318990,
    "belief": -6908820605821366843,
    "buffs": -6907330109749815048,
    "buildings": -6917203094771649915,
    "confirm": -6915627349826723809,
    "constructionMaterial": -6917180467021169596,
    "consumption": -6902845924876156586,
    "currentPatron": -6913489339087789708,
    "devotion": -6913943659840406396, # devotion/belief
    "discovery": -6913479974945708503,
    "download": -6912820157918341621,
    "effect": -6902234897441092053,
    "effects": -6902362035358284768,
    "eventDuration": -6902018417385309297,
    "extraGoods": -6900061796493286420,
    "festival": -6908773579491322283,
    "fuelEfficiency": -6901428646395682482,
    "global": -6910306885876902499,
    "globalEffects": -6900227375200473257,
    "goods": -6904656400857447148,
    "goodsConsumption": -6916926126237868583,
    "help": -6906640699676227597,
    "islandBuffs": -6901814024921012623,
    "islandWideEffects": -6911394270208630347,
    "language": -6910251369175148580,
    "needAttributes": -6913212391033157055,
    "needConsumption": -6901164581421416158,
    "needs": -6915455774919739315,
    "noPatron": -6899884938127726030,
    "outputStorage": -6913551300295748472,
    "patron": -6904053276046802194,
    "patronEffects": -6916892856177889166,
    "production": -6914202634429573508,
    "productionBuildings": -6914034826827989276,
    "productionChain": -6910138230344138817,
    "productivity": -6902990997164434871,
    "publicBuildings": -6899952445988598006,
    "residences": -6908559383606239043, # or -6907998875214561561, -6901535596004234866
    "residents": -6900991602074423381,
    "runningEvent": -6900991734456921008,
    "showInformation": -6915422247772755926,
    "showNeedsOfPopulationTier": -6911349978207385087,
    "silo": -6907773021498254428,
    "settings": -6909909211298253262,
    "traders": -6903619873875452438,
    "trading": -6911891323366335933,
    "total": -6912046064850205942,
    "tradeRoutes": -6915569607474692589,
    "venerationEffects": -6915910452867912431,
    "wonderEffect": -6905498542987856790,
    "world": -6907309872814266407,
    "workforce": -6914935202834947869
}.items():
    try:
        result.append({
            "name": referenceName,
            "lineID": lineID,
            "locaText": assets.texts.get(lineID).values
        })

        if referenceName == "global":
            for session in params["sessions"]:
                if session["guid"] == 37135:
                    session["locaText"] = result[-1]["locaText"]
                    break
    except:
        print(f"No localization found for {referenceName} with Line ID {lineID}")


add_parameter("texts", result, "Text", "Texts from the game with localization.")

In [32]:
assets.texts.get(-6902320995495941795).values

{'english': 'Effects',
 'german': 'Effekte',
 'spanish': 'Efectos',
 'french': 'Effets',
 'italian': 'Effetti',
 'japanese': '\u200b効果',
 'korean': '\u200b효과',
 'polish': 'Efekty',
 'brazilian': 'Efeitos',
 'russian': 'Эффект',
 'simplified_chinese': '\u200b效果',
 'traditional_chinese': '\u200b效果'}

In [33]:
assets.texts.get(-6902362035358284768).values

{'english': 'Effects',
 'german': 'Effekte',
 'spanish': 'Efectos',
 'french': 'Effets',
 'italian': 'Effetti',
 'japanese': '\u200b効果',
 'korean': '\u200b효과',
 'polish': 'Efekty',
 'brazilian': 'Efeitos',
 'russian': 'Эффект',
 'simplified_chinese': '\u200b效果',
 'traditional_chinese': '\u200b效果'}

## Ensure params is serializable

In [34]:
from typing import Dict
from assetextractor.parsing.core.attributes import Attribute


def find_unprocessed(data: t.Any, path: str = "root"):
    if isinstance(data, Attribute) or isinstance(data, Asset):
        raise ValueError(f"{path} is of type {data.__class__}")

    if isinstance(data, dict):
        for key in data:    
            if not isinstance(key, str):
                raise ValueError(f"{path}.{key} is not a string")     
            find_unprocessed(data[key], f"{path}.{key}")
    elif isinstance(data, list):
        for i, item in enumerate(data):
            find_unprocessed(item, f"{path}[{i}]")
    else:
        # Base case: primitive value
        pass

find_unprocessed(params)

### Save parameters and schema

In [35]:
with open("../anno-117-calculator/js/params.js", "w", encoding="utf-8") as f:
    f.write('if(window.params == null)window.params=')
    f.write(json.dumps(params, ensure_ascii=False, indent=2, sort_keys=True))

In [36]:

# Save schema to the same directory as params.js
with open("../anno-117-calculator/js/params.schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print("Finished")

Finished
